## TO do
POSITIONAL ENCODING

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import torch
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),                  
    transforms.Normalize((0.5,), (0.5,)),     
])

train_set = datasets.MNIST(
    root="./data",        
    train=True,
    download=True,
    transform=transform,
)

test_set = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=0)


In [4]:
train_set[0][0].shape

torch.Size([1, 28, 28])

In [5]:
class PatchEmbedding(nn.Module):
    """
    A layer that splits an image into patches and projects each patch to an embedding vector.
    Used as the input layer of a Vision Transformer (ViT).

    Inputs:
    - img_size: Integer representing the height/width of input image (assumes square image).
    - patch_size: Integer representing height/width of each patch (square patch).
    - in_channels: Number of input image channels
    - embed_dim: Dimension of the linear embedding space.
    """
    def __init__(self, img_size, patch_size, in_channels=1, embed_dim=64):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        assert img_size%patch_size==0, "img dim must be divisible by patch_size"
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim = in_channels * patch_size * patch_size
        self.proj = nn.Linear(self.patch_dim, embed_dim)
        self.embed_dim = embed_dim

    def forward(self, x):
        """
        input: x.shape = (B, C, H, W); C == in_channels
        
        patch_dim = C* patch_size * patch_size
        num_patches = (H//p) * (W//p)

        output: patchified img: (B, num_patches, embed_dim)
        """
        B, C, _, _ = x.shape
        p = self.patch_size
        x = x.view(B, C,  self.img_size//p, self.img_size//p, p, p)
        # print(x.is_contiguous())
        x = x.permute(0,3,1,2,4,5)
        # print(x.is_contiguous())

        x = x.reshape(B, self.num_patches, self.patch_dim)
        # print(x.is_contiguous())
        return self.proj(x)

In [6]:
## test
x = torch.randn(5, 1, 28, 28)
patching_test = PatchEmbedding(28, 7, 1, 64)
out = patching_test(x)
out.shape

torch.Size([5, 16, 64])

In [7]:
import math
def embedsinus(dim, x):
    """
    x -> (B,)
    """
    N=10000 #Attention is all you need  N**(k/(dim//2)) 
    h_dim = dim//2
    emb = math.log(N)/(h_dim)
    emb = torch.exp(torch.arange(h_dim) * -emb) # (h_dim,)  
    emb = x[:, None] * emb[None,:] #careful with broadcast
    return torch.cat((emb.sin(), emb.cos()), dim=-1)

def pe2d(num_patch, emb_dim):
    grid_size = int(num_patch**0.5)
    rows = torch.arange(grid_size)
    cols = torch.arange(grid_size)
    assert grid_size**2==num_patch, "num_patch has to be a square"
    assert emb_dim%4==0, "emb_dim has to be divisible by 4"
    rows = embedsinus(emb_dim // 2, rows)
    cols = embedsinus(emb_dim // 2, cols) 
    # what we do here is a bit more trickier than 1D 
    # we do a grid of positional encoding (col(i), row(j)) for the matrix/grid
    #we then flatten to get (1, num_patch, emb_dim)
    pe_r = rows[:, None, :].expand(grid_size, grid_size, emb_dim // 2)
    pe_c = cols[None, :, :].expand(grid_size, grid_size, emb_dim // 2)
    pe = torch.cat([pe_r, pe_c], dim=-1)
    print(pe.shape)
    return pe.view(1, num_patch, emb_dim)
embedsinus(16, torch.randn(5)).shape
# pe2d(9,16)

torch.Size([5, 16])

## Positional embedding

Pos embedding is trickier than it might seem. To understand why this works and how we do that, remember that we want to encode positional embedding which we can choose to be constant. We need something to describe positions and apparently 2 things seem important: the distance relation between 2 positions and the position itself, you can think of it as looking for a wide stationnary process?   
X(t) = (cos(ft), sin(ft)) where f is chosen.  
Now how exactly by just adding some values will the model "understands" the positional information. Key insight is what happens with attention where x' = x + pe, pe=pos encoding. Another insight is thinking of gradient flow: just a&dding some fixed input as pe will not influence gradient flow backward, but the forward values. Remember that (f(g(x)))' = f'(g(x))g'(x) with g(x) = x + pe, we still have g(x) appearing in the gradient values which then influents the model.
We destroy equivariance across values??


In [ ]:

class MultiHeadAtt(nn.Module):
    """
    MultiHeadAttention ??
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(embed_dim, embed_dim)
        self.query = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        
        self.attn_drop = nn.Dropout(dropout)
        assert embed_dim%num_heads==0, "emb_dim has to be divisible by num_heads"
        self.n_head = num_heads
        self.emb_dim = embed_dim
        self.head_dim = self.emb_dim // self.n_head

    def forward(self, query, key, value, attn_mask=None):
        """
        inputs: 
        query = (B, P, E)
        key = (B, T, E)
        value = (B, T, E') ? E=E' 
        P = patch_nums or sequence length 
        T = flexibile dim ? Target sequence
        E = emb_dim

        with T being target numbers of patch here for DiT? Target sequence length in NLP
        output: vect = (B, P, E) 
        Remember that we can choose depending on attention what do we put in query, key or value

        Self-attention: query=key=value=input
        Cross-attention: query=input, key=value=context

        """
        B, P, E = query.shape
        _, T, _ = key.shape

        queries = self.query(query).view(B, P, self.n_head, self.head_dim).transpose(1,2) # (B,P,E) -> (B, h, P, E//h)
        keys = self.key(key).view(B, T, self.n_head, self.head_dim).transpose(1,2) # (B, T, E) -> (B, h, T, E//h)
        values = self.value(value).view(B, T,self.n_head, self.head_dim).transpose(1,2) # (B, T, E') -> (B, h, T, E//h)

        # print(keys.shape, queries.shape)
        dot = queries @ keys.transpose(-2, -1)

        if attn_mask is not None:
          dot = dot.masked_fill_(attn_mask==0, float('-inf')) # in case I want to add some mask, but useless in DiT usually

        product = self.attn_drop(F.softmax(dot/np.sqrt(self.head_dim), -1)) # (B, h, P, T)
        out = product @ values # (B, h, P, T) @ (B, h, T, E//h) = (B, h, P, E//h)
        out = out.transpose(1,2).reshape(B, P, E) # here we need reshape or contiguous().view()
        return self.proj(out) 

# q = torch.randn(5, 16, 64)
# k = torch.randn(5, 8, 64)
# v =  torch.randn(5, 8, 64)
# att = MultiHeadAtt(64, 1)
# x = att(q, k, v)
# print(x.shape)

class MLP(nn.Module):
    """
    Designing a MLP 
    """
    def __init__(self, input_dim, dims):
        super().__init__()
        self.proj1 = nn.Linear(input_dim, dims[0])
        self.mods = nn.ModuleList([])
        dim_c = list(zip(dims[:-1], dims[1:]))
        for dim0, dim1 in dim_c :
            layer = nn.ModuleList([nn.SiLU(), nn.Linear(dim0, dim1) ])
            self.mods.append(layer)
    def forward(self, x):
        x = self.proj1(x)
        for layer in self.mods:
            act, fc = layer
            x = fc(act(x))
        return x
    

# con = MLP(5, [18, 26, 12])
# test_x_mlp = torch.randn(5,)
# print(con(test_x_mlp).shape)




    
class BlockadaLN(nn.Module):
    """
    DiT adaLN block of the DiT paper
    """
    def __init__(self,num_heads, emb_dim, cond_dim):
        super().__init__()
        self.selfatt = MultiHeadAtt(emb_dim, num_heads)
        self.mlp_cond = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, 6 * emb_dim))
        self.lnorm1 = nn.LayerNorm(emb_dim)
        # self.lnorm1 = nn.RMSNorm()
        self.lnorm2 = nn.LayerNorm(emb_dim)
        # self.lnorm2 = nn.RMSNorm()
        self.ffn = MLP(emb_dim, [4 * emb_dim, emb_dim])

    def forward(self, x, cond):
        """
        cond could be time only or both time and conditioning

        """
        sc1 = x
        x = self.lnorm1(x)
        cond = self.mlp_cond(cond)
        s1, s2, s3 = cond.chunk(3, dim=2)

        scale1, shift1 = s1.chunk(2, dim=2)
        x = x * (1 + scale1) + shift1
        x = self.selfatt(query=x, key=x, value=x)

        scale2, scale4 = s2.chunk(2, dim=2)
        x =  x * (1 + scale2)
        x = x + sc1
        sc2 = x
        x = self.lnorm2(x)
        
        scale3, shift3 = s3.chunk(2, dim=2)
        x = x * (1 + scale3) + shift3

        x = self.ffn(x) * (1 + scale4)
        return x + sc2

# x = torch.randn(8,16,32)
# cond = torch.randn(8, 16, 16)
# block_test = BlockadaLN(8, 32, 16)
# print(block_test(x, cond).shape)

        


torch.Size([8, 16, 32])


In [23]:
torch.randn(5,).shape

torch.Size([5])

## Questions

Why Q and K and not a single matrix M? 
- Low dimensionality - less parameters